# XGBoost from scratch (NumPy only)

What we build, following the lesson in `doc/Description.md`:

1. **Objective** = loss + regularisation: `Obj = sum L(y, yhat) + sum_k [gamma*T + lambda/2 * sum w^2 + alpha * sum |w|]`
2. **Second-order** (Newton) boosting: every sample carries a gradient `g` and hessian `h`.
3. **Optimal leaf weight** `w* = -G / (H + lambda)` and **split gain**
   `0.5 * [G_L^2/(H_L+lam) + G_R^2/(H_R+lam) - G^2/(H+lam)] - gamma`.
4. **Exact greedy** split finding (sort each feature, sweep with cumulative sums).
5. **Pre-pruning** via `gamma`, **shrinkage** (`eta`), **row / column subsampling**,
   **missing-value default directions**, early stopping, feature importance.
6. Losses: squared error and binary logistic (any twice-differentiable loss would plug in).

In [ ]:
import os
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("plots", exist_ok=True)
rng = np.random.default_rng(0)

## 1. Losses: gradient and hessian

For squared error `L = 1/2 (yhat - y)^2`: `g = yhat - y`, `h = 1`.

For logistic loss with raw score `f` and `p = sigmoid(f)`: `g = p - y`, `h = p (1 - p)`.
The hessian is small when the model is already confident (p near 0 or 1) - so those
samples get a *large* Newton step denominator weight only via `H + lambda`, i.e. a
leaf full of confident samples is not pushed around by a few noisy gradients.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


class SquaredError:
    name = "rmse"

    @staticmethod
    def base_score(y):
        return float(np.mean(y))

    @staticmethod
    def grad_hess(y, f):
        return f - y, np.ones_like(f)

    @staticmethod
    def transform(f):
        return f

    @staticmethod
    def metric(y, f):
        return float(np.sqrt(np.mean((y - f) ** 2)))


class Logistic:
    name = "logloss"

    @staticmethod
    def base_score(y):
        p = np.clip(np.mean(y), 1e-6, 1 - 1e-6)
        return float(np.log(p / (1 - p)))

    @staticmethod
    def grad_hess(y, f):
        p = sigmoid(f)
        return p - y, np.maximum(p * (1 - p), 1e-16)

    @staticmethod
    def transform(f):
        return sigmoid(f)

    @staticmethod
    def metric(y, f):
        p = np.clip(sigmoid(f), 1e-12, 1 - 1e-12)
        return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

## 2. One regularised tree (exact greedy)

For every feature we sort the node's samples once, take cumulative sums of `g` and `h`,
and get `G_L, H_L` for *every* threshold in a single vectorised pass. NaNs are handled the
XGBoost way: try sending all missing rows left, then right, and keep the better
direction (the **default direction**), stored with the split.

`alpha` (L1) enters through soft-thresholding of `G`: `T(G) = sign(G) * max(|G| - alpha, 0)`,
used in both the score and the leaf weight.

In [ ]:
class XGBTree:
    """A single regression tree grown depth-first with the XGBoost gain."""

    def __init__(self, max_depth=4, reg_lambda=1.0, reg_alpha=0.0, gamma=0.0,
                 min_child_weight=1.0, colsample=1.0, rng=None):
        self.max_depth, self.lam, self.alpha, self.gamma = max_depth, reg_lambda, reg_alpha, gamma
        self.min_child_weight, self.colsample = min_child_weight, colsample
        self.rng = rng if rng is not None else np.random.default_rng(0)
        # flat node arrays (filled while growing)
        self.feature, self.thr, self.left, self.right = [], [], [], []
        self.default_left, self.value, self.gain, self.cover = [], [], [], []

    # ---- helpers -------------------------------------------------------------
    def _T(self, G):
        return np.sign(G) * np.maximum(np.abs(G) - self.alpha, 0.0)

    def _score(self, G, H):
        return self._T(G) ** 2 / (H + self.lam)

    def leaf_weight(self, G, H):
        return -self._T(G) / (H + self.lam)

    def _new_node(self):
        for a in (self.feature, self.thr, self.left, self.right, self.default_left,
                  self.value, self.gain, self.cover):
            a.append(0)
        return len(self.feature) - 1

    # ---- exact greedy split search --------------------------------------------
    def _best_split(self, X, g, h, feats):
        G, H = g.sum(), h.sum()
        parent = self._score(G, H)
        best = dict(gain=-np.inf)
        for f in feats:
            x = X[:, f]
            miss = np.isnan(x)
            Gm, Hm = g[miss].sum(), h[miss].sum()
            xv, gv, hv = x[~miss], g[~miss], h[~miss]
            if xv.size < 2:
                continue
            order = np.argsort(xv, kind="stable")
            xs = xv[order]
            cg, ch = np.cumsum(gv[order]), np.cumsum(hv[order])
            valid = xs[:-1] < xs[1:]            # only split between distinct values
            if not valid.any():
                continue
            GL0, HL0 = cg[:-1], ch[:-1]
            for dleft in ((False, True) if miss.any() else (False,)):
                GL = GL0 + (Gm if dleft else 0.0)
                HL = HL0 + (Hm if dleft else 0.0)
                GR, HR = G - GL, H - HL
                ok = valid & (HL >= self.min_child_weight) & (HR >= self.min_child_weight)
                if not ok.any():
                    continue
                gain = 0.5 * (self._score(GL, HL) + self._score(GR, HR) - parent) - self.gamma
                gain = np.where(ok, gain, -np.inf)
                i = int(np.argmax(gain))
                if gain[i] > best["gain"]:
                    best = dict(gain=float(gain[i]), feature=int(f),
                                thr=float(0.5 * (xs[i] + xs[i + 1])), default_left=dleft)
        return best

    # ---- recursive growth -------------------------------------------------------
    def fit(self, X, g, h):
        self._grow(X, g, h, depth=0)
        return self

    def _grow(self, X, g, h, depth):
        nid = self._new_node()
        G, H = g.sum(), h.sum()
        self.value[nid] = float(self.leaf_weight(G, H))
        self.cover[nid] = float(H)
        if depth >= self.max_depth or len(g) < 2:
            return nid
        d = X.shape[1]
        k = max(1, int(round(self.colsample * d)))
        feats = self.rng.choice(d, size=k, replace=False) if k < d else np.arange(d)
        best = self._best_split(X, g, h, feats)
        # pre-pruning: gain already has -gamma subtracted; stop if not positive
        if not np.isfinite(best["gain"]) or best["gain"] <= 0:
            return nid
        f, t, dl = best["feature"], best["thr"], best["default_left"]
        xf = X[:, f]
        go_left = np.where(np.isnan(xf), dl, xf <= t)
        self.feature[nid], self.thr[nid], self.default_left[nid] = f, t, dl
        self.gain[nid] = best["gain"]
        self.left[nid] = self._grow(X[go_left], g[go_left], h[go_left], depth + 1)
        self.right[nid] = self._grow(X[~go_left], g[~go_left], h[~go_left], depth + 1)
        return nid

    @property
    def n_leaves(self):
        return sum(1 for l in self.left if l == 0)  # children are never node 0 (the root)

    def predict(self, X):
        out = np.zeros(len(X))
        self._predict(X, np.arange(len(X)), 0, out)
        return out

    def _predict(self, X, idx, nid, out):
        if self.left[nid] == 0:                       # leaf
            out[idx] = self.value[nid]
            return
        xf = X[idx, self.feature[nid]]
        go_left = np.where(np.isnan(xf), self.default_left[nid], xf <= self.thr[nid])
        self._predict(X, idx[go_left], self.left[nid], out)
        self._predict(X, idx[~go_left], self.right[nid], out)

    def importance(self, n_features):
        """Return (gain_sum, weight(#splits), cover_sum) per feature for this tree."""
        gs, ws, cs = (np.zeros(n_features) for _ in range(3))
        for nid, l in enumerate(self.left):
            if l != 0:
                f = self.feature[nid]
                gs[f] += self.gain[nid] + self.gamma   # raw objective improvement
                ws[f] += 1
                cs[f] += self.cover[nid]
        return gs, ws, cs

## 3. The boosting loop

```
F_0 = base score
for t in 1..T:
    g, h = loss.grad_hess(y, F)          # 2nd-order info at the current prediction
    (optionally subsample rows)
    tree_t = fit regularised tree on (g, h)
    F += eta * tree_t(X)                 # shrinkage
```
`second_order=False` replaces every hessian by 1 - this recovers *first-order* gradient
boosting (leaf = -mean gradient), handy to see what the Hessian buys us for logistic loss.

In [ ]:
class XGBoost:
    def __init__(self, objective="reg", n_estimators=100, learning_rate=0.1, max_depth=4,
                 reg_lambda=1.0, reg_alpha=0.0, gamma=0.0, min_child_weight=1.0,
                 subsample=1.0, colsample_bytree=1.0, second_order=True, seed=0):
        self.loss = SquaredError if objective == "reg" else Logistic
        self.n_estimators, self.eta = n_estimators, learning_rate
        self.tree_kw = dict(max_depth=max_depth, reg_lambda=reg_lambda, reg_alpha=reg_alpha,
                            gamma=gamma, min_child_weight=min_child_weight)
        self.subsample, self.colsample = subsample, colsample_bytree
        self.second_order, self.seed = second_order, seed

    def fit(self, X, y, X_val=None, y_val=None, early_stopping_rounds=None):
        rng = np.random.default_rng(self.seed)
        self.n_features = X.shape[1]
        self.base = self.loss.base_score(y)
        F = np.full(len(y), self.base)
        Fv = np.full(len(y_val), self.base) if X_val is not None else None
        self.trees, self.train_curve, self.val_curve = [], [], []
        best, best_it = np.inf, 0
        for t in range(self.n_estimators):
            g, h = self.loss.grad_hess(y, F)
            if not self.second_order:
                h = np.ones_like(h)
            idx = np.arange(len(y))
            if self.subsample < 1.0:
                idx = rng.choice(len(y), int(self.subsample * len(y)), replace=False)
            tree = XGBTree(colsample=self.colsample, rng=rng, **self.tree_kw)
            tree.fit(X[idx], g[idx], h[idx])
            self.trees.append(tree)
            F += self.eta * tree.predict(X)
            self.train_curve.append(self.loss.metric(y, F))
            if Fv is not None:
                Fv += self.eta * tree.predict(X_val)
                m = self.loss.metric(y_val, Fv)
                self.val_curve.append(m)
                if m < best - 1e-9:
                    best, best_it = m, t
                elif early_stopping_rounds and t - best_it >= early_stopping_rounds:
                    break
        self.best_iteration = best_it if Fv is not None else len(self.trees) - 1
        return self

    def raw_predict(self, X, n_trees=None):
        F = np.full(len(X), self.base)
        for tr in self.trees[: (n_trees or len(self.trees))]:
            F += self.eta * tr.predict(X)
        return F

    def predict(self, X, n_trees=None):
        return self.loss.transform(self.raw_predict(X, n_trees))

    def feature_importance(self, kind="gain"):
        gs, ws, cs = (np.zeros(self.n_features) for _ in range(3))
        for tr in self.trees:
            a, b, c = tr.importance(self.n_features)
            gs += a; ws += b; cs += c
        if kind == "weight":
            return ws
        if kind == "cover":
            return np.where(ws > 0, cs / np.maximum(ws, 1), 0)   # average cover per split
        return np.where(ws > 0, gs / np.maximum(ws, 1), 0)       # average gain per split

## 4. Sanity check of the maths on a toy node

Verify the closed-form leaf weight really minimises the second-order objective
`G w + 1/2 (H + lambda) w^2` for a leaf (numerically brute-forced).

In [ ]:
g_toy = np.array([0.5, -0.2, 0.8, 0.3]); h_toy = np.array([0.25, 0.2, 0.24, 0.21])
lam = 1.0
G_, H_ = g_toy.sum(), h_toy.sum()
w_star = -G_ / (H_ + lam)
ws = np.linspace(-3, 3, 100001)
obj = G_ * ws + 0.5 * (H_ + lam) * ws ** 2
print(f"closed-form w* = {w_star:.5f}   brute-force argmin = {ws[np.argmin(obj)]:.5f}")
print(f"objective at w*: {-0.5 * G_**2 / (H_ + lam):.5f} (= -1/2 G^2/(H+lambda))")

## 5. Regression demo: a noisy 1-D function

Effect of the number of rounds and of `gamma`/`lambda` on the fitted curve.

In [ ]:
n = 300
x = rng.uniform(-3, 3, n)
y_reg = np.sin(2 * x) + 0.3 * x + rng.normal(0, 0.3, n)
Xr = x[:, None]
xt = np.linspace(-3, 3, 400)[:, None]

configs = [
    ("5 trees, eta=0.3",            dict(n_estimators=5, learning_rate=0.3)),
    ("50 trees, eta=0.1",           dict(n_estimators=50, learning_rate=0.1)),
    ("200 trees, eta=0.3, lam=0",   dict(n_estimators=200, learning_rate=0.3, reg_lambda=0.0, min_child_weight=0)),
    ("200 trees, eta=0.3, lam=20, gamma=1", dict(n_estimators=200, learning_rate=0.3, reg_lambda=20.0, gamma=1.0)),
]
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8), sharey=True)
for ax, (title, kw) in zip(axes, configs):
    m = XGBoost("reg", max_depth=4, seed=1, **kw).fit(Xr, y_reg)
    ax.scatter(x, y_reg, s=8, c="gray", alpha=0.6)
    ax.plot(xt[:, 0], np.sin(2 * xt[:, 0]) + 0.3 * xt[:, 0], "g--", lw=1, label="truth")
    ax.plot(xt[:, 0], m.predict(xt), "r", lw=1.8, label="XGB")
    ax.set_title(title, fontsize=9)
axes[0].legend()
plt.tight_layout(); plt.savefig("plots/01_regression_fit.png", dpi=110); plt.close()
print("saved plots/01_regression_fit.png")

## 6. Regularisation knobs: `lambda` and `gamma`

* `lambda` shrinks leaf weights (visible in the *max |leaf weight|*) and dampens splits of
  low-hessian nodes.
* `gamma` is a hard bar on gain: bigger `gamma` -> fewer leaves (pre-pruning).

We fit a **single** tree so the effect on structure is directly visible, then check
validation error of a full ensemble.

In [ ]:
Xa = rng.normal(size=(1500, 8))
ya = np.sin(Xa[:, 0] * 2) + Xa[:, 1] * Xa[:, 2] + 0.5 * (Xa[:, 3] > 0) + rng.normal(0, 0.5, 1500)
Xa_tr, Xa_va, ya_tr, ya_va = Xa[:1000], Xa[1000:], ya[:1000], ya[1000:]

def one_tree(lam, gam):
    g, h = SquaredError.grad_hess(ya_tr, np.full(len(ya_tr), ya_tr.mean()))
    t = XGBTree(max_depth=8, reg_lambda=lam, gamma=gam, min_child_weight=0.0).fit(Xa_tr, g, h)
    return t.n_leaves, np.max(np.abs(t.value))

gammas = [0, 0.5, 1, 2, 5, 10, 20, 50]
lams = [0, 1, 10, 100]
leaves_gamma = [one_tree(1.0, gm)[0] for gm in gammas]
maxw_lam = [one_tree(lm, 0.0)[1] for lm in lams]

val_by_lam = {}
for lm in lams:
    m = XGBoost("reg", n_estimators=150, learning_rate=0.1, max_depth=6, reg_lambda=lm,
                min_child_weight=0.0).fit(Xa_tr, ya_tr, Xa_va, ya_va)
    val_by_lam[lm] = m.val_curve

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(gammas, leaves_gamma, "o-"); ax[0].set_xscale("symlog", linthresh=0.5)
ax[0].set_xlabel("gamma"); ax[0].set_ylabel("#leaves (1 tree, depth<=8)"); ax[0].set_title("gamma prunes the tree")
ax[1].plot(lams, maxw_lam, "o-"); ax[1].set_xscale("symlog", linthresh=1)
ax[1].set_xlabel("lambda"); ax[1].set_ylabel("max |leaf weight|"); ax[1].set_title("lambda shrinks leaf weights")
for lm, c in val_by_lam.items():
    ax[2].plot(c, label=f"lambda={lm}")
ax[2].set_xlabel("boosting round"); ax[2].set_ylabel("validation RMSE"); ax[2].legend()
ax[2].set_title("depth-6 ensemble, min_child_weight=0")
plt.tight_layout(); plt.savefig("plots/02_lambda_gamma.png", dpi=110); plt.close()
print("leaves vs gamma:", dict(zip(gammas, leaves_gamma)))
print("final val RMSE by lambda:", {k: round(v[-1], 4) for k, v in val_by_lam.items()})

## 7. Binary classification (logistic loss)

Synthetic 10-feature problem with a non-linear boundary, 3 informative + noise features.
We use early stopping on a validation split and compare:

* **second-order** (Newton) vs **first-order** (hessian := 1) boosting,
* small vs large learning rate (`eta` vs number of trees trade-off).

In [ ]:
def make_cls(n, seed):
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, 10))
    z = 1.5 * X[:, 0] * X[:, 1] + np.sin(2 * X[:, 2]) + 0.8 * X[:, 3] ** 2 - 1.0
    y = (r.random(n) < sigmoid(2 * z)).astype(float)
    return X, y

Xtr, ytr = make_cls(2500, 1)
Xva, yva = make_cls(1000, 2)
Xte, yte = make_cls(2000, 3)
acc = lambda y, p: float(np.mean((p > 0.5) == y))

t0 = time.time()
m_newton = XGBoost("clf", n_estimators=300, learning_rate=0.1, max_depth=4, subsample=0.8,
                   colsample_bytree=0.8, reg_lambda=1.0, seed=0).fit(Xtr, ytr, Xva, yva, early_stopping_rounds=25)
print(f"second-order: best iter {m_newton.best_iteration}, test acc "
      f"{acc(yte, m_newton.predict(Xte, m_newton.best_iteration + 1)):.4f}  ({time.time() - t0:.1f}s)")
m_first = XGBoost("clf", n_estimators=300, learning_rate=0.1, max_depth=4, subsample=0.8,
                  colsample_bytree=0.8, reg_lambda=1.0, second_order=False, seed=0).fit(Xtr, ytr, Xva, yva, early_stopping_rounds=25)
print(f"first-order : best iter {m_first.best_iteration}, test acc "
      f"{acc(yte, m_first.predict(Xte, m_first.best_iteration + 1)):.4f}")
m_hi = XGBoost("clf", n_estimators=300, learning_rate=0.4, max_depth=4, subsample=0.8,
               colsample_bytree=0.8, seed=0).fit(Xtr, ytr, Xva, yva, early_stopping_rounds=25)
m_lo = XGBoost("clf", n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8,
               colsample_bytree=0.8, seed=0).fit(Xtr, ytr, Xva, yva, early_stopping_rounds=25)
for nm, m in (("eta=0.4", m_hi), ("eta=0.05", m_lo)):
    print(f"{nm}: best iter {m.best_iteration}, best val logloss {min(m.val_curve):.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(m_newton.train_curve, "b--", label="2nd-order train")
ax[0].plot(m_newton.val_curve, "b", label="2nd-order val")
ax[0].plot(m_first.train_curve, "r--", label="1st-order train")
ax[0].plot(m_first.val_curve, "r", label="1st-order val")
ax[0].axvline(m_newton.best_iteration, c="b", ls=":")
ax[0].set_xlabel("round"); ax[0].set_ylabel("logloss"); ax[0].legend(); ax[0].set_title("Newton vs first-order boosting")
for nm, m, c in (("eta=0.4", m_hi, "C1"), ("eta=0.1", m_newton, "C0"), ("eta=0.05", m_lo, "C2")):
    ax[1].plot(m.val_curve, c, label=f"{nm} (stop@{m.best_iteration})")
ax[1].set_xlabel("round"); ax[1].set_ylabel("validation logloss"); ax[1].legend()
ax[1].set_title("learning rate vs #trees (early stopping)")
plt.tight_layout(); plt.savefig("plots/03_logistic_curves.png", dpi=110); plt.close()

## 8. Feature importance: gain vs weight vs cover

Features 0-3 carry signal, 4-9 are noise. Gain-based importance separates them best;
`weight` (split counts) is inflated for noisy features that get chosen a lot at deep,
low-gain nodes.

In [ ]:
imp = {k: m_newton.feature_importance(k) for k in ("gain", "weight", "cover")}
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for a, (k, v) in zip(ax, imp.items()):
    a.bar(range(10), v / max(v.sum(), 1e-12), color=["C3"] * 4 + ["gray"] * 6)
    a.set_title(f"importance: {k} (normalised)"); a.set_xlabel("feature (0-3 informative)")
plt.tight_layout(); plt.savefig("plots/04_feature_importance.png", dpi=110); plt.close()
print("gain importance:", np.round(imp["gain"] / imp["gain"].sum(), 3))

## 9. Missing values & sparsity-aware default directions

We knock out 30% of feature 0 **informatively** (rows with y=1 are more likely to be
missing). No imputation: each split learns whether NaNs should go left or right.
Compare with a mean-imputation baseline (same booster).

In [ ]:
Xn_tr, Xn_te = Xtr.copy(), Xte.copy()
def knock_out(X, y, seed):
    r = np.random.default_rng(seed)
    p_miss = np.where(y == 1, 0.5, 0.1)
    X = X.copy(); X[r.random(len(y)) < p_miss, 0] = np.nan
    return X
Xn_tr, Xn_te = knock_out(Xtr, ytr, 5), knock_out(Xte, yte, 6)
Xn_va = knock_out(Xva, yva, 7)

m_nan = XGBoost("clf", n_estimators=120, learning_rate=0.1, max_depth=4, seed=0).fit(Xn_tr, ytr, Xn_va, yva)
mu = np.nanmean(Xn_tr[:, 0])
imp_ = lambda X: np.where(np.isnan(X), mu, X)
Xi_tr, Xi_va, Xi_te = imp_(Xn_tr), imp_(Xn_va), imp_(Xn_te)
m_imp = XGBoost("clf", n_estimators=120, learning_rate=0.1, max_depth=4, seed=0).fit(Xi_tr, ytr, Xi_va, yva)
print(f"native NaN handling test acc: {acc(yte, m_nan.predict(Xn_te)):.4f}")
print(f"mean-imputation     test acc: {acc(yte, m_imp.predict(Xi_te)):.4f}")
dirs = [(t.default_left[i]) for t in m_nan.trees for i, f in enumerate(t.feature) if t.left[i] != 0 and f == 0]
print(f"splits on feature 0: {len(dirs)}, NaN sent left in {np.mean(dirs):.0%} of them")

## 10. Timing / scale note and summary

Exact greedy costs `O(n log n)` per feature per node (sorting). LightGBM (next folder)
replaces this with histograms. Here we just report our training time.

In [ ]:
t0 = time.time()
Xbig, ybig = make_cls(6000, 11)
XGBoost("clf", n_estimators=20, max_depth=5).fit(Xbig, ybig)
print(f"20 trees, n=6000, d=10, depth 5, exact greedy: {time.time() - t0:.2f}s")
print("\nSummary: test accuracy (logistic, early-stopped) =",
      round(acc(yte, m_newton.predict(Xte, m_newton.best_iteration + 1)), 4))